# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from rich.pretty import pprint

import json2vec as j2v

In [2]:
records = pl.DataFrame(
    {
        "color": ["red", "orange", "yellow", "blue", "green", "purple"],
        "label": ["warm", "warm", "warm", "cool", "cool", "cool"],
    }
)

In [3]:
params = j2v.schema(
    j2v.Column("color", j2v.Category, max_vocab_size=16),
    j2v.Column("label", j2v.Category, target=True, max_vocab_size=8, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
)


model = j2v.Architecture(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    processor=None,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)


2026-05-22 15:17:25.394 | INFO     | json2vec.architecture.root:__init__:121 - initialized JSON2Vec module


In [4]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=datamodule)


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argume

In [5]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [6]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [0.9976333975791931, 0.9971972703933716],
│   │   │   'null': [0.00037397799314931035, 0.0004505390825215727],
│   │   │   'padded': [0.0002542748406995088, 0.00028356563416309655],
│   │   │   'masked': [0.0014754768926650286, 0.001790917944163084],
│   │   │   'other': [0.0002628520014695823, 0.0002778882335405797]
│   │   },
│   │   'content': {
│   │   │   'value': ['cool', 'cool'],
│   │   │   'probability': [0.6815276145935059, 0.715091347694397],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.6815276145935059},
│   │   │   │   │   {'label': 'warm', 'probability': 0.3184725046157837}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.715091347694397},
│   │   │   │   │   {'label': 'warm', 'probability': 0.284908652305603}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [7]:
params.set(j2v.where("name") == "record", embed=True)

pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.3068569600582123,
│   │   │   │   -0.14279218018054962,
│   │   │   │   0.2257833331823349,
│   │   │   │   -0.2463705688714981,
│   │   │   │   -0.01932661421597004,
│   │   │   │   -0.29937243461608887,
│   │   │   │   -0.30184850096702576,
│   │   │   │   -0.24751880764961243,
│   │   │   │   0.28068608045578003,
│   │   │   │   -0.35711780190467834,
│   │   │   │   0.09231782704591751,
│   │   │   │   0.2627248466014862,
│   │   │   │   0.02657967619597912,
│   │   │   │   -0.13383005559444427,
│   │   │   │   0.19599497318267822,
│   │   │   │   0.43646466732025146
│   │   │   ],
│   │   │   [
│   │   │   │   0.2525314390659332,
│   │   │   │   -0.23795630037784576,
│   │   │   │   0.2309773713350296,
│   │   │   │   -0.24097099900245667,
│   │   │   │   -0.11450657993555069,
│   │   │   │   -0.2631613612174988,
│   │   │   │   -0.2689087688922882,
│   │   │   │   -0.22530236840248108,
│   │   │   │   0.3581351637840271,
│   │   │   │   -0.26060453057289124,
│   │   │   │   0.060648415237665176,
│   │   │   │   0.24584415555000305,
│   │   │   │   0.016377754509449005,
│   │   │   │   -0.16607025265693665,
│   │   │   │   0.21671296656131744,
│   │   │   │   0.4771195352077484
│   │   │   ]
│   │   ]
│   }
}

In [8]:
model.plot(detail=True)

╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ n_heads: 4                                                                                                           │
│ d_model: 16                                                                                                          │
│ parameters: 14454                                                                                                    │
│                                                                                                                      │
│   ┏━ record (array) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│   ┃ embed: True                                                                                                    ┃ │
│   ┃ n_heads: 4                                                                                                     ┃ │
│   ┃ attention: mha                                                                                                 ┃ │
│   ┃ max_length: 1                                                                                                  ┃ │
│   ┃ n_outputs: 1                                                                                                   ┃ │
│   ┃ n_linear: 1                                                                                                    ┃ │
│   ┃ n_layers: 1                                                                                                    ┃ │
│   ┃ parameters: 6608                                                                                               ┃ │
│   ┃                                                                                                                ┃ │
│   ┃   ╭─ color (category) ───────────────────────────────────────────────────────────────────────────────────────╮ ┃ │
│   ┃   │ address: record/color                                                                                    │ ┃ │
│   ┃   │ embed: False                                                                                             │ ┃ │
│   ┃   │ n_heads: 4                                                                                               │ ┃ │
│   ┃   │ query: [*].color                                                                                         │ ┃ │
│   ┃   │ pooling: query                                                                                           │ ┃ │
│   ┃   │ weight: 1.0                                                                                              │ ┃ │
│   ┃   │ n_linear: 1                                                                                              │ ┃ │
│   ┃   │ max_vocab_size: 16                                                                                       │ ┃ │
│   ┃   │ n_bands: 8                                                                                               │ ┃ │
│   ┃   │ p_unavailable: 0.01                                                                                      │ ┃ │
│   ┃   │ topk: []                                                                                                 │ ┃ │
│   ┃   │ parameters: 4055                                                                                         │ ┃ │
│   ┃   │                                                                                                          │ ┃ │
│   ┃   │ state                                                                                                    │ ┃ │
│   ┃   │   vocabulary_size: 6                                                                                     │ ┃ │
│   ┃   │   vocabulary: [yellow, green, blue, purple, red, orange]                                                 │ ┃ │
│   ┃   │                                                                                                          │ ┃ │
│   ┃  

'<!DOCTYPE html>\n<html>\n<head>\n<meta charset="UTF-8">\n<style>\n\nbody {\n    color: #000000;\n    background-color: #ffffff;\n}\n</style>\n</head>\n<body>\n    <pre style="font-family:Menlo,\'DejaVu Sans Mono\',consolas,\'Courier New\',monospace"><code style="font-family:inherit">╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮\n│ n_heads: 4                                                                                                           │\n│ d_model: 16                                                                                                          │\n│ parameters: 14454                                                                                                    │\n│                                                                                                                      │\n│   ┏━ record (array) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━